# COMP560 ReID — Colab Runner

Fill in the **Config** cell, then Run All.

Pipeline:
1. Mount Drive (checkpoints + datasets shared)
2. Clone repo
3. Copy / symlink dataset_a images and Market-1501 from Drive
4. Install deps
5. Run `train.py --config configs/experiments/<CONFIG_NAME>.yaml`
6. Copy `checkpoints/<exp_id>/` back to shared Drive folder

In [ ]:
# ====== Config ======
CONFIG_NAME   = 'exp01_teacher_dinov2_large'  # change per run
REPO_URL      = 'https://github.com/<YOUR_ORG>/object-reid.git'  # TODO: fill in
BRANCH        = 'main'

# Shared Drive folder (must exist and contain images_dataset_a.zip + market1501.zip)
DRIVE_SHARED  = '/content/drive/MyDrive/comp560_reid_shared'

# Local working directory on Colab
WORKDIR       = '/content/object-reid'

# Where to push trained checkpoints back to
DRIVE_CKPT_DIR = f'{DRIVE_SHARED}/checkpoints'

print('CONFIG_NAME:', CONFIG_NAME)

In [ ]:
# ====== Mount Drive ======
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.isdir(DRIVE_SHARED), f'Shared folder not found: {DRIVE_SHARED}'
print(os.listdir(DRIVE_SHARED))

In [ ]:
# ====== Clone repo ======
import subprocess, os, shutil
if os.path.isdir(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, WORKDIR])
os.chdir(WORKDIR)
!git log -1 --oneline

In [ ]:
# ====== Install deps ======
!pip install -q -r requirements.txt

In [ ]:
# ====== Unpack datasets from Drive ======
import os, shutil
os.makedirs('datasets/dataset_a', exist_ok=True)
os.makedirs('datasets/market1501', exist_ok=True)

# dataset_a images
if not os.path.isdir('datasets/dataset_a/images'):
    !unzip -q "$DRIVE_SHARED/images_dataset_a.zip" -d datasets/dataset_a

# dataset_a parquets (our_train/our_test/test)
for name in ['our_train.parquet', 'our_test.parquet', 'test.parquet', 'train.parquet']:
    src = os.path.join(DRIVE_SHARED, name)
    dst = os.path.join('datasets/dataset_a', name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

# Market-1501 (expects prepared parquets; otherwise run prepare_market1501.py)
if not os.path.isdir('datasets/market1501/bounding_box_train'):
    !unzip -q "$DRIVE_SHARED/market1501.zip" -d datasets/market1501

for name in ['market1501_train.parquet', 'market1501_test.parquet']:
    src = os.path.join(DRIVE_SHARED, name)
    dst = os.path.join('datasets/market1501', name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

# If parquets weren't in Drive, generate them now
if not os.path.exists('datasets/market1501/market1501_train.parquet'):
    !python scripts/prepare_market1501.py --market_root datasets/market1501

print('dataset_a:', os.listdir('datasets/dataset_a'))
print('market1501:', os.listdir('datasets/market1501')[:10])

In [ ]:
# ====== (Optional) pull teacher checkpoint from Drive for KD runs ======
import os, shutil
teacher_src = f'{DRIVE_CKPT_DIR}/exp01_teacher_dinov2_large/best.pth'
teacher_dst = 'checkpoints/exp01_teacher_dinov2_large/best.pth'
if os.path.exists(teacher_src):
    os.makedirs(os.path.dirname(teacher_dst), exist_ok=True)
    shutil.copy(teacher_src, teacher_dst)
    print('Teacher loaded:', teacher_dst)
else:
    print('No teacher at', teacher_src, '(OK if this is not a KD run)')

In [ ]:
# ====== Train ======
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python train.py --config configs/experiments/$CONFIG_NAME.yaml

In [ ]:
# ====== Copy checkpoints back to Drive ======
import os, shutil
src = f'checkpoints/{CONFIG_NAME}'
dst = f'{DRIVE_CKPT_DIR}/{CONFIG_NAME}'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
if os.path.isdir(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('Uploaded to', dst)
!ls -lh "$dst"